In [ ]:
from typing import Any, List, Optional, Callable
from jaxtyping import Array, Float, Int, PyTree # https://github.com/google/jaxtyping

import dataclasses

import jax
import jax.numpy as jnp
import numpy as np
import distrax
print("Distrax version:", distrax.__version__)
import flax
from flax import nnx
print("Flax version:", flax.__version__)

Distrax version: 0.1.8
Flax version: 0.12.7


In [ ]:
class MLP(nnx.Module):
    def __init__(self,
                 layer_dims: Sequence[tuple],
                 activation: Callable = nnx.leaky_relu,
                 use_bias: bool = True,
                 rngs: nnx.Rngs = nnx.Rngs(0),
                 kernel_init: Callable = nnx.initializers.lecun_normal()
                 ):
                 
        assert len(layer_dims)>0, "At least one layer dimension must be specified in layer_dims."
        self.activation = activation
        self.layer_dims = tuple(tuple(x) if not isinstance(x, tuple) else x for x in layer_dims)
        self.layers = nnx.List()
        for layer_dim in layer_dims:
            self.layers.append(nnx.Linear(layer_dim[0], 
                                          layer_dim[1], 
                                          rngs=rngs, 
                                          use_bias=use_bias, 
                                          kernel_init=kernel_init
                                          )
                                )
    def __call__(self, x):
        for l, layer in enumerate(self.layers[:-1]):
            x = self.activation(layer(x))
        x = self.layers[-1](x)
        return x
        
    

class Conditioner(nnx.Module):
    def __init__(self,
                 features_shape: tuple,
                 context_shape: tuple,
                 layer_dims: Sequence[tuple],
                 num_bijector_params: int,
                 activation: Callable = nnx.leaky_relu,
                 ):
        self.features_shape = features_shape
        self.context_shape = context_shape
        self.layer_dims = layer_dims
        self.num_bijector_params = num_bijector_params
        self.activation = activation

        self.n_flat_features = jax.tree.reduce(lambda carry, x: carry*x, features_shape, start=1)
        self.n_flat_context = jax.tree.reduce(lambda carry, x: carry*x, context_shape, start=1)
        self._conditioner = nnx.List()
        self._conditioner_mlp.append(MLP(layer_dims, activation)) # Build the NN as specified by layer_dims that learns the sline transformation parameters
        self._conditioner_out.append(nnx.Linear(layer_dimes[-1][-1]),
                                 self.n_flat_features*num_bijector_params,
                                 activation=activation,
                                 )
    
    def __call__(self):
        x = self._conditioner_mlp(x)
        x = self.activation(x)
        x = self._conditioner_out(x)
        return x
        # Stack feature and context vectors and pass through MLP


class RQSplineFlow(nnx.Module):
    layers: nnx.List

    def __init__(self,
                 n_dim: int,
                 n_context: int = 0
                 n_transforms: int = 4
                 hidden_dims: List[int] = dataclasses.field(default_factory=lambda: [128, 128])
                 activation: str = "gelu"
                 n_bins: int = 8
                 range_min: float = -1.0
                 range_max: float = 1.0
                 event_shape: Optional[List[int]] = None
                 context_shape: Optional[List[int]] = None
                 ):

                 self.layers = []
    
    def __call__(self):
        for layer in self.layers:
            x = layer(x)
        return x

